# 篇 1 · 倾斜与热点 key：一个 task 跑了 90% 时间，怎么在这台机器上量出来

> 要回答的坏结果：**"shuffle 之后有一个 task 比其它 task 慢一到两个数量级"**。
> 本篇不写"salting 能治倾斜"这种口号，而是：造一个**确定的** 40% 落在同一个 key 上的事实表 →
> 把"一个分区占大头"量成一个数 → 三种解法**各治哪一半** → 用同一组数量出差异。
>
> **结论押在确定量上，耗时只打印。** 本篇所有断言都是"同一镜像上每次跑都给同一个数"的那一类
> （最大桶行数占比、两集合之差、计划里的算子名、计划里有没有 skew 痕迹）；三种写法的墙上时间**一条都不 assert**。
> 原因见 ④：本机是 `local[*]` 系列（本镜像的 kernel 是 `local[2]`，② 那一步会打出来），
> 倾斜的代价在这里体现不出来，硬断言只会得到一条与结论无关的红。
>
> ## 本机能演示到哪一层（先说破，别拿本地的绿当分布式的证据）
>
> - `local[2]` 是**一个 JVM 里的 2 个线程**：没有节点、没有 executor 崩溃、没有 `FetchFailedException`。
>   "某台机器被热点 task 拖垮 → 上游重试 → 整条链路变慢"这类失效模式**本机演示不了**，本篇凡涉及它都写「未验证」。
> - 没有真正的网络 shuffle：数据在同一个进程里换分区，`spark.shuffle.io.*`、
>   `maxTotalFileSize`、本地盘那一层都碰不到。
> - 内存是 driver 的 512MB（kernel.json 里写死的 `--driver-memory`），
>   所以 broadcast 撑爆 executor 这种事也只能讲口径、不能量。
> - 数据与配置全部当场生成：`spark.range` + 确定性表达式，**不落任何文件进 git**，
>   notebook 也**不带 outputs**（结果属于执行的那一刻；抄进 git 就再也追不回它是哪天、哪个镜像跑出来的）。
>
> 内核请选 `PySpark (arena)`（`metadata.kernelspec.name` = `arena-pyspark`）——
> 交付前的闸门 `server/test/notebooks/tutorials.test.ts` 会**真执行这一篇**并按内核名判：
> 写成 `python3` 会跑在镜像自带的系统解释器上，那正是本项目划在外面的那条隔离红线。
> 最后一个单元是 `spark.stop()`：闸门跑完不留活内核 —— 这句话**有判据**（`tutorials.test.ts` 常驻那组的
> 结构性那条，判的就是"每篇教程最后一个 code cell 里有 `spark.stop()`"，宿主档 45 秒就能红）。

*本文里出现的每个数字都是这一篇在本容器（Spark 3.5.5 / `local[2]` / 100 万行）实测出来的；
从头执行一遍实测 26 秒（含 Spark 冷启动）。跨版本行为一律标「未在本镜像验证」。*


In [ ]:
import re
import time

from pyspark.sql import SparkSession, functions as f

spark = SparkSession.builder.appName("wi90-01-skew").getOrCreate()

# ⑤ 那张速查表里的默认值不是背来的，是这里读的：SET -v 打的是**这个镜像**的生效值。
# 顺序很重要 —— 下面两行覆盖之前先读一次，否则"默认值"读出来的是我自己改过的值。
# CITE 的准入规矩（复审 M4 立的）：**⑤ 表里敢写"实测"的那个键，就必须出现在这里打印的行里** ——
# 不然"实测"只是作者的记忆，读者跑一遍看不见，那就成了纪律二禁止的"把没验的写成已验"。
CITE = [
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.advisoryPartitionSizeInBytes",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.broadcastTimeout",
    "spark.sql.files.maxPartitionBytes",
]
_defaults = {str(r["key"]): str(r["value"]) for r in spark.sql("SET -v").collect()}
print("出厂生效值（本镜像）")
for k in CITE:
    print("   ", k, "=", _defaults.get(k, "(这个版本没有这个键)"))

spark.conf.set("spark.sql.shuffle.partitions", 8)
spark.conf.set("spark.sql.adaptive.enabled", False)  # 先关 AQE：② 要量的是"没有它时"的坏形状（前提在 ② 里断言）
print("本篇改成", spark.conf.get("spark.sql.shuffle.partitions"), "个 shuffle 分区 | adaptive =", spark.conf.get("spark.sql.adaptive.enabled"))
print("spark", spark.version, "| master", spark.sparkContext.master, "| defaultParallelism", spark.sparkContext.defaultParallelism)


## ① 现象：造一个「40% 的行落在同一个 key 上」的事实表

真实形状是这样的：匿名流量里"未登录用户"共用一个哨兵 id、某个大 V 占了全站互动的一成、
`shop_id = 0` 代表"没有店铺"。共同点：**一个 key 吃掉一大块行**，而 `groupBy` 的 hash 分区
只会看 key 的哈希值 —— 同一个 key 的所有行必然进同一个桶，那个桶就是一个 task。

下面这段把这件事**确定地**造出来（不是"随机出来大概是这样"）：

```python
user_id = "HOT"                   if id % 5 < 2   else  "u" + str(id % 199_000)
```

- `id % 5 < 2` 命中 2/5 ⇒ **恰好 40% 的行**共用 `HOT` 这一个 key（② 会把它量出来，不是照抄这句话）。
- `amount` 必须 `.cast("long")`：**浮点求和与相加顺序有关**，而 ④ 那条判据是
  "加盐前后的结果**逐位相等**"。用 double，那条断言会因为分区顺序不同而红，
  红得毫无道理 —— 这是本篇最想提醒你的一次踩坑。
- 数据里没有一处随机，所以**不需要 seed**。顺带一个 API 事实（实测）：
  `SparkSession.range()` **没有 `seed` 参数**（`range(start, end, step, numPartitions)`），
  写 `spark.range(N, seed=7)` 直接 `TypeError`。seed 是 `sample()` / `F.rand(seed)` 那一族的参数。
  教程纪律里的"随机数据固定 seed"在这里由**确定性表达式**满足 —— 换 seed 也换不出新形状。


In [ ]:
N = 1_000_000
events = (
    spark.range(N)
    .withColumn(
        "user_id",
        f.when(f.col("id") % 5 < 2, f.lit("HOT"))  # 恰好 40% 落到一个 key
        .otherwise(f.concat(f.lit("u"), (f.col("id") % 199_000).cast("string"))),
    )
    .withColumn("amount", (f.col("id") % 100).cast("long"))  # long：两种聚合顺序之和必须逐位相等
)
events.createOrReplaceTempView("events")

t = time.time()
rows = events.count()
print("行数", rows, "| 输入分区数", events.rdd.getNumPartitions(), "| 造数据耗时", round(time.time() - t, 2), "s")

t = time.time()
keys = events.select("user_id").distinct().count()
print("不同 key 数", keys, "|", round(time.time() - t, 2), "s")


## ② 取证：把「一个分区占大头」量成一个数

**度量对象是那次 hash shuffle 的输入，不是聚合的结果。** 这句话是整篇的技术核心，
也是最容易写错的地方，所以两种量法都跑一遍，让数字自己说话：

| 量法 | 量的东西 | 本机实测最大占比 | 能判倾斜吗 |
| --- | --- | --- | --- |
| ❌ 对 `groupBy().agg()` 的结果算分区行数占比 | 聚合**之后**：每个 key 只剩 1 行 | 0.1270（≈ 1/8，八个桶几乎平分） | 不能 —— 倾斜已经被聚合压平了 |
| ✅ 对 `repartition(8, "user_id")` 的输出算行数占比 | 聚合**之前**：每个桶要吃多少行 | 0.4754 | 能，这就是倾斜 |

第一行不是稻草人：它是本篇的第一版写法，`assert shares_before[0] > 0.30` 因此永远红。
倾斜发生在"行被 hash 到桶里"那一步，聚合一旦做完信息就没了。

第二行为什么可信：`repartition(8, "user_id")` 产生的 Exchange 和 `groupBy("user_id").agg(...)`
要做的 Exchange 是**同一个** `hashpartitioning(user_id, 8)` —— 下面那个 `hash_partitioning_of()`
就把两行计划文本取出来逐字比一下，比不过就红。这一步不是装饰：代理度量最容易坏在
"我量的那个 shuffle 和真查询里那个不是同一个"。

计划文本走 `df._jdf.queryExecution().simpleString()`（实测可用）。**别**用
`df.explain()` 的返回值做断言 —— 它 `print` 到 stdout 然后返回 `None`；
也别用 `_jdf.executedPlan()` / `_jdf.physicalPlan()` —— 本镜像实测 `Py4JError: Method executedPlan([]) does not exist`。

**② ③ ④ 各单元末尾那行 `WI90[...] OK` 是给闸门读的**（`server/test/notebooks/tutorial-claims.ts` 登记了本篇四条结论，
容器档 `tutorials.test.ts` 按**双向相等**核对）。破坏性验证实测过两种坏法：
把"加盐把最大桶降到一半以下"那条 `assert` 连同它的 print 一起删掉 ⇒ 执行层一路绿、结论层红在
"没打出这些结论标记"（只看"没报错"就是不够的那一半）；把 `id % 5 < 2` 改成 `id % 5 == 0`（20%）⇒ 红在
`HOT 的占比只有 0.2 ⇒ 现象不成立`。所以那行 print 不是装饰，别删也别改格式。

复审之后**又实测了三种新的坏法**（差别在于：它们不需要真跑 Spark，宿主档那 45 秒就会红 ——
判据读的是这份 notebook 的**源码字节**）：

| 坏法 | 红在哪一条 | 实测到的红话（摘要） |
| --- | --- | --- |
| 只删 `assert`、**留着**那行 marker print | 「每条已登记 marker 所在的那个 code cell 里必须至少有一条 assert」 | 「打印 marker 的那个 code cell（本篇第 5 个 code cell）里**一条语句级 assert 都没有** ⇒ 那行「…[salting-preserves-result] OK」现在是**没有证据的自报**…改的是教程正文…不是删这条判据、也不是删 marker」 |
| 四条 marker 塞进一个 `for` 循环用 f-string 拼出来 print | 同一条的前半句 | 「marker 字面量「WI90[…][hot-key-dominates] OK」在**任何** code cell 源码里都不存在 ⇒ …②print 被挪进循环或函数里用 f-string 拼出来 —— 执行层看到的输出是对的，但**闸门读的是字节**…」 |
| 删掉最后一个单元的 `spark.stop()` | 「每篇教程的最后一个 code cell 必须 spark.stop()」 | 「…导语那句"闸门跑完不留活内核"就成了假话，且容器档下一篇教程会多背一个活着的 Spark JVM…改的是教程…不是删这条判据」 |

第一条补的是结论层最后一个没人管的洞：**删 assert 留 print** 时执行层零异常、marker 双向相等全过，
整档绿着替一条没人判的结论作证。第二条判死"把 marker 变成一个循环" —— 那种写法在 stdout 上一字不差，
所以判据只能看字节。它的**射程**也要说清：只判"那个单元里有没有人在判"，
**不判阈值对不对**（那半边按裁决留给交付档）。


In [ ]:
SHUFFLE_PARTITIONS = 8  # 与 spark.sql.shuffle.partitions 同一个数：② 量的就是"那 8 个桶"


def max_partition_share(df, expected_partitions=SHUFFLE_PARTITIONS):
    """每个 shuffle 桶的行数占比（降序）+ 每桶行数 + 总行数。

    它不看耗时，所以在同一镜像上每次都给同一个数 —— 这就是本篇把结论押在它身上、
    而不押"task 时长占比"的原因（时长那条要等真的分布式才有形状，见开头边界）。
    桶数写进判据：度量对象"是 8 个桶"这件事属于结论内容，分区被改写过的话占比就没有意义了。
    """
    counts = df.rdd.mapPartitions(lambda it: iter([sum(1 for _ in it)])).collect()
    assert len(counts) == expected_partitions, (
        f"度量对象应有 {expected_partitions} 个桶，实际 {len(counts)} 个 ⇒ 分区数被改写了"
        "（AQE 的 coalescePartitions 合并？还是 shuffle.partitions 被这一段之外的人改了？）—— 红在这里别去改占比阈值"
    )
    total = sum(counts)
    return sorted((c / total for c in counts), reverse=True), counts, total


def hash_partitionings_of(df):
    """物理计划里**全部** Exchange 的 hashpartitioning(键, 桶数），按计划文本顺序（外层在前），
    归一化掉属性 id（user_id#2 → user_id）。

    为什么要有"复数"这一版（复审 m2）：嵌套计划（加盐两阶段的第二阶段）的文本里天然有**两次**
    hash shuffle —— 它自己那一次 + 被它包在下面的第一阶段。那种地方"恰好一次"的前提不成立，
    只能按位置比，所以取外层那一次（下标 0）来核对。
    """
    lines = [ln.strip() for ln in df._jdf.queryExecution().simpleString().splitlines() if "hashpartitioning(" in ln]
    return ["hashpartitioning(" + re.sub(r"#\d+", "", ln.split("hashpartitioning(", 1)[1].split("),", 1)[0]) + ")" for ln in lines]


def hash_partitioning_of(df):
    """取那个**唯一的** hash shuffle。多条 Exchange 的计划请改用 `hash_partitionings_of()`。"""
    lines = hash_partitionings_of(df)
    assert len(lines) == 1, f"期望恰好一次 hash shuffle，实际 {lines}"
    return lines[0]


# 前提：AQE 必须真的关着。① 的度量定义里"8 个桶"是内容的一部分，
# 而 AQE 的 coalescePartitions 会换掉桶数（下面解法 C 那段有实测：200 个 shuffle 分区开 AQE 只落 2 个文件）。
# 走 df.rdd 这条路本机实测不会被合并，但前提不能靠运气，所以写成断言。
assert spark.conf.get("spark.sql.adaptive.enabled") == "false", (
    f"spark.sql.adaptive.enabled = {spark.conf.get('spark.sql.adaptive.enabled')} —— ② 的度量前提是关掉 AQE。"
    "红在这里说的是**上面那行配置被删了/改回去了**，不是倾斜消失了：别改这条断言，把配置加回去"
)

plain = events.groupBy("user_id").agg(f.sum("amount").alias("amount"), f.count(f.lit(1)).alias("n"))

# ✅ 正确的度量对象：那次 Exchange 的输入
shuffle_input = events.repartition(SHUFFLE_PARTITIONS, "user_id")
assert hash_partitioning_of(shuffle_input) == hash_partitioning_of(plain), (
    f"代理度量的分区方式与真查询不是一个：{hash_partitioning_of(shuffle_input)} vs {hash_partitioning_of(plain)}"
)
shares_before, counts_before, total_before = max_partition_share(shuffle_input)

# ❌ 那条错量法，跑给读者看它对倾斜多不敏感
shares_of_aggregated, _, total_of_aggregated = max_partition_share(plain)

t = time.time()
hot_share = events.filter(f.col("user_id") == "HOT").count() / N
print("HOT 这个 key 占全表行数比例", round(hot_share, 4), "|", round(time.time() - t, 2), "s")
print("每桶行数", counts_before, "总计", total_before)
print("每桶行数占比（降序前 5）", [round(s, 4) for s in shares_before[:5]])
print("[对照·错量法] 聚合结果每桶占比（降序前 5）", [round(s, 4) for s in shares_of_aggregated[:5]], "行数", total_of_aggregated)
print("真查询的 Exchange 行", hash_partitioning_of(plain), "| 代理的", hash_partitioning_of(shuffle_input))

assert hot_share > 0.35, f"HOT 的占比只有 {hot_share} ⇒ 现象不成立，去查 user_id 那列的表达式，别改阈值"
assert shares_before[0] > 0.30, f"最大桶只占 {shares_before[0]} ⇒ 8 个桶是平分的，这里没有倾斜可讲"
assert shares_of_aggregated[0] < 0.20, f"聚合结果的最大桶占比 {shares_of_aggregated[0]}：倾斜已经被压平了，这条对照就是②要的东西"
print("WI90[01-skew-and-hot-keys][hot-key-dominates] OK")


## ③ 解法 A：加盐两阶段聚合（治**聚合**那一半）

```
第一阶段：groupBy(user_id, salt)   →  一个 key 的 400,000 行被拆成 16 份，进 16 组不同的桶
第二阶段：groupBy(user_id)          →  每组的 1 行部分结果再收敛，HOT 这一次只有 16 行
```

`salt` 用 `id % 16` —— 确定性、不需要额外一列、也不会改变语义（同一 key 的行被均匀切 16 刀，
每刀的 `sum`/`count` 相加还是原来的那个数）。

**各治哪一半，先说清楚：**

| 解法 | 治 | 不治 | 代价 |
| --- | --- | --- | --- |
| 加盐两阶段（本节） | **聚合**的倾斜（`groupBy` / `agg`） | join 的倾斜（那要另一套） | 多一次 shuffle；只对**可分解**的聚合成立 |
| 广播小维表（解法 B） | **join** 的倾斜 | 聚合的倾斜（groupBy 没有可广播的 build 侧） | build 侧要整个进内存 |
| AQE skew join（解法 C） | join 的倾斜（且要够大才触发） | 聚合的倾斜 | 只在 shuffle 统计真的倾斜时生效 |

"只对可分解的聚合成立"这句是加盐的边界，面试里追问一层就落到这里：
`sum` / `count` / `min` / `max` 可以先局部再全局；
**精确 `count(distinct)` 不行** —— 第一阶段没法把"去重"拆开，加盐只会让同一个值在 16 个桶里各数一遍。
精确去重要改成两阶段，而且**盐不许进到去重那一层的分组键里**：
先 `groupBy(维度, distinct_col).count()` 把去重做完（一个值一行，并行度靠的是**分区数**，不是盐），
再 `groupBy(维度).count()` 数行数 —— 那就是题库 `bd-pyspark-0001` 的题面本体（⑤ 下面有指针），
也与本篇 5.3 第 2 条、与题库参考答案写的是同一个形状。

⚠ 把盐写成 `groupBy(维度, distinct_col, salt)` 是**系统性高估**（复审 C1 抓的就是这一句的早期版本）：
同一个 `(维度, distinct_col)` 会因 `id % SALT` 不同被拆进好几个中间组，而第二阶段的 `count()`
把每个组各数一次 ⇒ 一个用户被数了至多 SALT 次。真要在去重层加盐，第二阶段就得换成
`count_distinct(distinct_col)` 而不是 `count()` —— **那是另一套形状，本篇没有跑它**，
按纪律二标「未验证」，别把它当"照抄就能用"的结论。


In [ ]:
SALT = 16
salted = events.withColumn("salt", f.col("id") % SALT)
stage1 = salted.groupBy("user_id", "salt").agg(f.sum("amount").alias("amount"), f.count(f.lit(1)).alias("n"))
unsalted = stage1.groupBy("user_id").agg(f.sum("amount").alias("amount"), f.sum("n").alias("n"))

# 量的还是"真计划里那次 shuffle"：第一阶段 Exchange 的输入（同一个 hashpartitioning 逐字核过）
salted_shuffle = salted.repartition(SHUFFLE_PARTITIONS, "user_id", "salt")
assert hash_partitioning_of(salted_shuffle) == hash_partitioning_of(stage1), (
    f"加盐前后的度量对象不是同一个 shuffle：{hash_partitioning_of(salted_shuffle)} vs {hash_partitioning_of(stage1)}"
)
shares_after, counts_after, _ = max_partition_share(salted_shuffle)

# 第二次 shuffle 的输入：HOT 在这里只剩 16 行（第一阶段的输出），所以它不再倾斜
stage2_proxy = stage1.repartition(SHUFFLE_PARTITIONS, "user_id")
# 本篇自己立的规矩（② 那句"代理度量最容易坏在我量的那个 shuffle 和真查询里那个不是同一个"）
# 在这一处也走一遍：代理量的那次 Exchange 必须**就是**第二阶段 groupBy 要做的 exchange（复审 m2）。
# 用复数版取外层那一次，是因为 stage1 是嵌套计划 —— 它自己那层 Exchange 也在同一份文本里。
assert hash_partitionings_of(stage2_proxy)[0] == hash_partitionings_of(unsalted)[0] == hash_partitioning_of(shuffle_input), (
    f"第二次 shuffle 的代理度量与真查询不是同一个 Exchange：代理外层 {hash_partitionings_of(stage2_proxy)} vs "
    f"真计划 {hash_partitionings_of(unsalted)}（期望两边最外层都是 {hash_partitioning_of(shuffle_input)}）"
    " ⇒ 下面那条『中间态不再倾斜』量的就不是第二阶段；红在这里改的是度量对象，别去改 shares_stage2 的阈值"
)
shares_stage2, counts_stage2, total_stage2 = max_partition_share(stage2_proxy)

print("加盐后每桶行数", counts_after, "总计", total_before)
print("加盐后每桶行数占比（降序前 5）", [round(s, 4) for s in shares_after[:5]])
print("最大桶占比：加盐前", round(shares_before[0], 4), "→ 加盐后", round(shares_after[0], 4), f"（降到 {shares_after[0] / shares_before[0]:.1%}）")
print("第二次 shuffle 的输入行数", total_stage2, "| 每桶占比（降序前 5）", [round(s, 4) for s in shares_stage2[:5]])

assert shares_after[0] < shares_before[0] / 2, (
    f"加盐后最大桶占比 {shares_after[0]} 没降到 {shares_before[0] / 2} 以下 ⇒ 盐没起作用，去查 salt 是不是进了分组键"
)
assert shares_stage2[0] < 0.20, f"第二次 shuffle 的最大桶占比 {shares_stage2[0]} —— 中间态不该再倾斜了"
print("WI90[01-skew-and-hot-keys][salting-halves-max-partition] OK")


## ④ 量差异，第一件是**答案没变**，不是"看起来更快"

加盐这类改写最危险的坏法不是没效果，而是**效果有、答案错了**：第一阶段少算/重算一个桶，
屏幕上还是"最大占比降了 3 倍"，读的人却拿着一份错的 UV 去汇报。所以这一节把
"加盐不许改变答案"排在耗时前面，而且是本篇最硬的一条断言（三个集合差都必须是 0）：

- `only_plain` / `only_salted`：两种写法的 **key 集合**必须一模一样（左反对称差）。
- `mismatch`：key 相同而 `amount` 或 `n` 不同 —— 这条抓的是"值算错了"。

`amount` 是 `long`（① 里那句 `.cast("long")` 就是为了这里）：整数加法满足结合律，
两种分区顺序必须逐位相等；换成 `double` 之后这条会红，而红的原因是浮点顺序，不是逻辑错。

耗时对照放在本节最后一个单元（只打印）。


In [ ]:
t = time.time()
only_plain = plain.join(unsalted, ["user_id"], "left_anti").count()
only_salted = unsalted.join(plain, ["user_id"], "left_anti").count()
mismatch = (
    plain.join(unsalted, ["user_id"], "inner")
    .filter((plain.amount != unsalted.amount) | (plain.n != unsalted.n))
    .count()
)
print("结果行数 plain", plain.count(), "| salted", unsalted.count(), "|", round(time.time() - t, 2), "s")
print("只在一侧的 key", only_plain, only_salted, "| 值不一致的 key", mismatch)

assert (only_plain, only_salted, mismatch) == (0, 0, 0), (only_plain, only_salted, mismatch)
print("WI90[01-skew-and-hot-keys][salting-preserves-result] OK")


## ③ 解法 B：广播小维表，绕开 join 的 shuffle（治 **join** 那一半）

join 倾斜的根因和聚合是同一句话：`SortMergeJoin` 要求**两侧都按 join key 做 hash 分区**，
于是热点 key 的行在两侧都挤进同一个桶、同一个 task。
`BroadcastHashJoin` 不走这条路 —— 它把小表复制进每个分区，**join 不再按 key 分区**，
热点 key 的行留在它们原来的分区里被就地处理。这就是"广播绕开倾斜"的机制，
也解释了它为什么治不了聚合：`groupBy` 没有"另一侧"可以搬。

代价说清楚：build 侧要整个进内存（默认阈值 10MB，最前面那个单元 `CITE` 打印的出厂值就是这个数），
超了就是 driver/executor OOM 或 `spark.sql.broadcastTimeout`（出厂值同样由 `CITE` 打印，别背，跑一遍就有）；
小表"以为小"是最常见的一类生产事故 —— 维度表有多版本时会先膨胀再广播，
那就是 `bd-pyspark-0002` 那道题（要先收敛维度再广播）。

⚠ 这一节还有一个 API 坑值得单独记：**物理计划是 lazy val**。
改完 `autoBroadcastJoinThreshold` 再对**同一个 DataFrame 对象**取一次计划，
文本一字不变 —— 本篇第一版就是这样把"广播改变了计划"判成假的。
正确写法是每种阈值**新建一个 DataFrame**（下面那个 `join_plan_at()` 就是为这件事存在的）。


In [ ]:
dim = spark.range(50).select(f.concat(f.lit("u"), f.col("id").cast("string")).alias("user_id"), f.col("id").alias("dim_v"))
keys = events.select("user_id").distinct()  # ② 里量出来的那 119,401 个 key


def join_plan_at(threshold):
    """按给定阈值取一次 join 的物理计划文本。

    ⚠ 每次都要**新建 DataFrame**：`queryExecution` 的 executedPlan 是 lazy val，
    复用同一个对象的话阈值改了计划也不会重新选策略（本机实测：文本一字不变）。
    """
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", threshold)
    j = keys.join(dim, "user_id", "left")
    return j._jdf.queryExecution().simpleString()


plan_off = join_plan_at(-1)  # 关掉自动广播
plan_on = join_plan_at(10 * 1024 * 1024)  # 放开到 10MB（本镜像的出厂默认值）

print("关掉广播阈值的 join 算子：", [ln.strip() for ln in plan_off.splitlines() if "Join" in ln])
print("放开广播阈值的 join 算子：", [ln.strip() for ln in plan_on.splitlines() if "Join" in ln])
print("放开之后有没有 BroadcastExchange：", "BroadcastExchange" in plan_on)

assert "SortMergeJoin" in plan_off, "阈值关了还不走 sort-merge ⇒ 本机数据形状与假设不符，别改断言，看上面那行计划"
assert "Broadcast" not in plan_off, "阈值关了还广播 ⇒ 有 hint 或 AQE 在背后改写，这一篇的前提不成立"
assert "BroadcastHashJoin" in plan_on, "阈值放开了没换成 BroadcastHashJoin ⇒ 小表没被认成小表（看上面的计划行）"
assert "SortMergeJoin" not in plan_on, "换策略没换干净：一份计划里两种 join"
print("WI90[01-skew-and-hot-keys][broadcast-changes-plan] OK")

t = time.time()
print("真的跑一次广播 join：结果行数", keys.join(dim, "user_id", "left").count(), "|", round(time.time() - t, 2), "s")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)  # 恢复出厂值（开头 CITE 打印的那个），别把这段的阈值留给后面


## ③ 解法 C：AQE（skew join / 分区合并）—— 这一节对**"AQE 有没有把倾斜救回来"一条断言都不加**，这是本篇最重要的一段

先给结论，因为它是"不需要时间也成立"的那一句：

> **skew hint 与 AQE 的 skew join 都只救 join，救不了聚合。**
> 机制层的原因：聚合的倾斜发生在 `hashpartitioning(聚合键, n)` 那一步 —— 一个 key 只映射到一个桶，
> 而"把这个桶劈成两半分给两个 task"会直接改变分组语义（同一个 key 的部分结果落在两个分区，
> 谁也不知道自己是不是完整的）。join 不一样：倾斜的那个 key 的**另一侧**行可以复制到多个分区补齐，
> `OptimizeSkewedJoin` 就是把倾斜分区的一侧拆成几块、另一块侧做多次小 join，语义不变。
> 所以 Spark 给 join 留了自动改写，给聚合只留了"你自己加盐"。

这一节**只对"AQE 有没有把倾斜真正拆开"这件事不断言**，两条理由都是实测的：

1. **AQE 的 skew join 有触发门槛，本机这个数据规模到不了。** 判定是
   `分区字节 > median × skewedPartitionFactor` **且** `> skewedPartitionThresholdInBytes`，
   本镜像 `SET -v` 实测是 5.0 与 **256MB**（这两个数由下面那段打印出来，不是背的；**网上大量资料写 512MB，
   那是从哪个版本起的默认值本篇没有逐版本核实**，迁移时以官方配置页为准）。下面那段把倾斜桶写成 parquet 数了字节：**最大桶 2.59MB**，差两个数量级 ⇒
   本机跑一百年也不会触发。把"skew join 生效了"写成断言，就是一条随机红。
2. **`hint("skew")` 在这个版本上不是聚合的解药，甚至连 join 的解药都不算。**
   实测三种写法（`dim.hint("skew")`、`dim.hint("skew", "user_id")`、SQL 里
   `/*+ SKEW(a.user_id) */`，与下面代码里那三处**一字不差**）**都不出现在物理计划里**，JVM 侧会打
   `WARN HintErrorLogger: Unrecognized hint: skew(...)`。那几行**确实会出现在这个单元的输出里**，
   但它走的是 kernel 的 **stderr** 流（渲染在 Jupyter 里看得见），而闸门的 marker 只读 stdout ——
   所以本篇不拿它当判据对象（stderr 的格式与措辞随版本漂，用它做断言就是给自己埋一条随机红）。
   3.5.5 有消费者的 hint 只有 broadcast / merge / shuffle_hash / shuffle_replicate_nested_loop
   这几个 join 策略 hint；**4.0 是否支持 skew hint：未在本镜像验证**（依据：官方 SQL hint 文档与 4.0 release notes）。

**但上面第 2 条里"都不出现在物理计划里"这半句是确定量，所以它现在有一条断言**（复审 m3 补的）：
下面那个单元把三种写法的计划痕迹数成 `hint_traces` 并 `assert hint_traces == [0, 0, 0]`。
它不给 marker（结论层的四条不变），坏了就走执行层 —— 症状是"这一篇有 cell 抛异常"，
红话说的是**本节正文过期了**，不是"AQE 生效了"。为什么值得断：教程里最容易被背走的一句就是
"skew hint 没用"，而它是个会随版本翻脸的事实；没断言时它只有一行 print 兜着，print 不挡任何人改正文。

顺带把 AQE 三段能力的存在与否量一下（下面那段跑的是真的）：
`coalescePartitions` 会把 shuffle 分区**合并** —— 本镜像实测把 `shuffle.partitions=200`
写出 **200 个文件**（AQE 关）vs **2 个文件**（AQE 开，2 = `local[2]` 的 defaultParallelism）。
这就是 ② 为什么要显式关掉 AQE 并把它写成前提断言：度量对象里"8 个桶"是结论内容的一部分。
（键名以本镜像 `SET -v` 打印出来的那个为准：这里印的是 `spark.sql.adaptive.advisoryPartitionSizeInBytes`。
**其它版本有没有改过名，本篇没有逐版本核实** —— 迁移时以官方配置页为准，别把这份教程的版本史当依据背。）


In [ ]:
import os
import shutil
import tempfile

spark.conf.set("spark.sql.adaptive.enabled", True)
setv = {str(r["key"]): str(r["value"]) for r in spark.sql("SET -v").collect()}
SKEW_KEYS = [
    "spark.sql.adaptive.enabled",
    "spark.sql.adaptive.skewJoin.enabled",
    "spark.sql.adaptive.skewJoin.skewedPartitionFactor",
    "spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes",
    "spark.sql.adaptive.forceOptimizeSkewedJoin",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize",
    # ⑤ 表里那一行写着"与 parallelismFirst 配合才有意义"，所以这个键必须在这里打印出来（复审 M4）：
    # 表里敢说"实测"的键，就得有一行打印撑着，否则那两个字只是作者的记忆。
    "spark.sql.adaptive.coalescePartitions.parallelismFirst",
    "spark.sql.adaptive.optimizeSkewsInRebalancePartitions.enabled",
]
print("AQE 相关生效值（本镜像 SET -v 实测）")
for k in SKEW_KEYS:
    print("   ", k, "=", setv.get(k, "(这个版本没有这个键)"))

# 1) skew hint 有没有进计划：三种写法各取一次物理计划，只数带 skew 痕迹的行
def skew_traces(df):
    """物理计划文本里带 skew 痕迹的行数（0 = 没有任何算子消费这个 hint）。

    数的子串是 **"kew"** 而不是 "skew"：Spark 的算子名大小写两种都有
    （`OptimizeSkewedJoin` / `AQESkewedJoinWithCE` 是大写 S，配置键与小写写法是小 s），
    取公共的那三个字母就一次抓全，而物理计划里没有第二个以 "kew" 出现的词，所以不会误抓。
    """
    return sum(1 for ln in df._jdf.queryExecution().simpleString().splitlines() if "kew" in ln)


plain.createOrReplaceTempView("events_agg")  # 给下面 SQL 注释那种写法用（views 是本篇唯一的"表"）
big_left = events.groupBy("user_id").agg(f.sum("amount").alias("amount"))
skew_sql = spark.sql("SELECT /*+ SKEW(a.user_id) */ a.user_id, b.amount FROM (SELECT DISTINCT user_id FROM events) a JOIN events_agg b ON a.user_id = b.user_id")
hint_traces = [
    skew_traces(big_left.join(dim.hint("skew"), "user_id", "left")),
    skew_traces(big_left.join(dim.hint("skew", "user_id"), "user_id", "left")),
    skew_traces(skew_sql),
]
print("hint 在物理计划里的痕迹行数（三种写法各一，0 = 没被任何算子消费）：", hint_traces)
print("（0 = 没有任何算子消费它；同一个单元的输出里应当有 WARN HintErrorLogger: Unrecognized hint: skew(...) —— 那是 stderr，本教程不拿它当判据）")
# 复审 m3：这一节"不加断言"说的是「AQE 有没有真把倾斜拆开」那件**要规模/要时间**才有的事，
# 而"三种写法的 hint 都没进物理计划"是**确定量**（同一份计划文本每次都给同一个数），断得起。
# 断它的好处是把"文档漂移"变成一次红：哪天这个版本的 hint 有消费者了，正文那句"都不出现在物理计划里"
# 就必须跟着改 —— 那正是本篇最容易被背错的一句话。不给它加 marker（结论层由 ②③④ 那四条 gate）。
assert hint_traces == [0, 0, 0], (
    f"三处 skew hint 的计划痕迹不再是全 0（现在是 {hint_traces}）⇒ **改的是本节正文**："
    '解法 C 那句「都不出现在物理计划里」与 5.4 那条口径都要跟着重写，'
    "因为教程教的是这个版本的行为，而它变了。别删这条断言、也别删 hint 那三行来让它闭嘴"
)

# 2) 倾斜桶到底多少字节：写成 parquet 数文件（这是 skewJoin 阈值的判据对象）
spark.conf.set("spark.sql.adaptive.enabled", False)
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
out = tempfile.mkdtemp(prefix="wi90-01-bucket-")
t = time.time()
shuffle_input.write.mode("overwrite").parquet(out)
sizes = sorted((os.path.getsize(os.path.join(out, fn)) for fn in os.listdir(out) if fn.endswith(".parquet")), reverse=True)
print("每桶落盘字节（降序前 4）", sizes[:4], "总计", sum(sizes), "|", round(time.time() - t, 2), "s")
shutil.rmtree(out, ignore_errors=True)

# 3) AQE 的分区合并真的会动桶数：同一句聚合写两次
spark.conf.set("spark.sql.shuffle.partitions", 200)
paths = {}
for label, adaptive in (("AQE 关", False), ("AQE 开", True)):
    spark.conf.set("spark.sql.adaptive.enabled", adaptive)
    p = tempfile.mkdtemp(prefix=f"wi90-01-{label}-")
    t = time.time()
    plain.write.mode("overwrite").parquet(p)
    paths[label] = (len([fn for fn in os.listdir(p) if fn.endswith(".parquet")]), round(time.time() - t, 2))
    shutil.rmtree(p, ignore_errors=True)
print("shuffle.partitions=200 时写出的文件数：", paths)

# 恢复本篇的前提
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
spark.conf.set("spark.sql.adaptive.enabled", False)
print("恢复后：shuffle 分区", spark.conf.get("spark.sql.shuffle.partitions"), "| adaptive", spark.conf.get("spark.sql.adaptive.enabled"))


## ④ 量差异，第二件：耗时**只打印**

这一段是本篇唯一**专门做耗时对照**的地方（前面各段里那些 `time.time()` 只是给每一步取数/落盘打个收尾数字，
不拿来互相比较），而它大概率会让"加盐更快"这句话在这里**不成立**：
`local[2]` 只有一个 JVM、两个线程、没有网络 shuffle、8 个桶每桶几 MB，
调度开销主导一切 ⇒ 实测顺序经常是 `plain` 最快、`salted` 因为多一次 shuffle 而更慢。

那为什么还写？因为读者必须看见这个反差才能记住本篇的主张：
**倾斜的代价是"一个 task 拖住一个节点 + 上游 fetch 失败重试"，那是节点级的失效模式，
本机测不到（未验证）。** 本机能量到的是**形状**（桶的行数占比）和**正确性**（集合差为 0），
所以断言全部押在这两条上。把 `assert t_salted < t_plain` 写进教程，
得到的不是结论，是一条每次跑都可能翻转的红 —— 这类断言本仓库一律拒收。

三条对照（同一份数据、同一个 8 桶设置）：

| 写法 | 分区键 | 断言了吗 | 想说明什么 |
| --- | --- | --- | --- |
| A `groupBy(user_id)` @8 | user_id | ②③④ 都断过 | 基线：一个桶吃 47.5% 的行 |
| B 加盐两阶段 @8 | (user_id, salt) → user_id | ③④ 都断过 | 最大桶降到 17.5%，答案一字不变 |
| C `groupBy(user_id)` @100 | user_id | 占比**断过**（`assert shares_100[0] > 0.30`）；耗时**只打印** | 分区数从 8 加到 100，最大桶还是 40.6% —— 加分区不解决热点 key |

C 那一行是本篇最常被误解的地方：`spark.sql.shuffle.partitions` 决定的是**桶的个数**，
而"同一个 key 进同一个桶"是 hash 分区规则本身。想验证就照着下面那段跑。
这一行的占比是个确定量，所以复审之后把它断上了（以前这里只 print，而表格写着"形状那条断过"——
读者会以为"加分区不解决热点 key"这件事被判过，其实判的是 8 桶那条）；**耗时那一栏仍然只打印**，
同一条单元里两类东西各有归宿，这才是这张表想教的事。

另外：本篇**不 `cache()`**。一 cache，后面三种写法的计时量的就不再是那两次 shuffle 了 ——
教程里"数字变漂亮了"最常见的来源就是这个。


In [ ]:
def timed(label, thunk):
    """跑一次并把耗时打出来。**不返回断言**：见上面那段，本机时间不构成结论。"""
    t = time.time()
    r = thunk()
    print(f"{label}: {round(time.time() - t, 2)} s -> {r}")
    return r


spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)
timed("A plain groupBy @8   行数", lambda: plain.count())
timed("B salted 两阶段 @8   行数", lambda: unsalted.count())

# C：把分区数调大，倾斜的形状一模一样（这条是确定量 ⇒ 断得起；但不为它加 marker，见下面的注释）
spark.conf.set("spark.sql.shuffle.partitions", 100)
timed("C plain groupBy @100 行数", lambda: events.groupBy("user_id").agg(f.sum("amount").alias("amount"), f.count(f.lit(1)).alias("n")).count())
shares_100, counts_100, _ = max_partition_share(events.repartition(100, "user_id"), expected_partitions=100)
print("100 个桶的最大占比", round(shares_100[0], 4), f"（8 个桶时是 {round(shares_before[0], 4)}）| 桶数", len(counts_100))
# 复审 M5：占比是确定量，断它零成本。以前这里只 print，而上面那张表写着"形状那条断过"，
# 读者会以为"加分区不解决热点 key"这件事被判过 —— 判的其实是 8 桶那一条。现在断在这一处。
# 不新增 slug（结论层仍是那四条）：加了 marker 就必须同时登记进 tutorial-claims.ts，
# 只改一边会红在"打出了注册表里没有的结论标记"或"没打出这些结论标记"。
assert shares_100[0] > 0.30, (
    f"100 个桶时最大桶只占 {shares_100[0]} ⇒「加分区不解决热点 key」这条不成立，"
    "去查 user_id 那列的表达式与 hash 分区规则是不是被这段之外的人改了 —— 红在这里别改阈值"
)
spark.conf.set("spark.sql.shuffle.partitions", SHUFFLE_PARTITIONS)

# 两边答案的全表汇总（同一个数才是"没改变答案"的第二遍证据）
timed("plain  全表 sum/count", lambda: plain.agg(f.sum("amount"), f.count_distinct("user_id")).collect()[0])
timed("salted 全表 sum/count", lambda: unsalted.agg(f.sum("amount"), f.count_distinct("user_id")).collect()[0])

print("汇总一下本篇的两个确定量：最大桶占比", round(shares_before[0], 4), "->", round(shares_after[0], 4), "| HOT 占比", round(hot_share, 4))


## ⑤ 速查表（可以只靠它回答问题）+ 追问链

### 5.1 线上有个 task 跑了 40 分钟，先查哪三个数

按这个顺序查，因为第一个最便宜且**不受机器负载影响**：

| # | 查什么 | 怎么查 | 判什么 |
| --- | --- | --- | --- |
| 1 | **每个 shuffle 桶的行数占比** | 本篇 ② 的 `max_partition_share()`（或 Spark UI 那个 stage 的 Tasks 表按 **TaskGCTime/records** 排序看最大那条） | 最大一条 ≫ 1/分区数 ⇒ **键的倾斜**，继续查 2 |
| 2 | **每个桶的字节数** vs 中位数 | 落盘数文件（本篇解法 C 那段）/ UI 的 shuffled size | `> 5 × median` 且 `> 256MB` ⇒ AQE 的 skew join **有可能**自动管（join 才有）；够不着阈值就别指望它 |
| 3 | **那个 stage 里最慢 task 在做什么** | UI: Summary metrics（spill disk/memory、GC time、input records vs output records） | 慢 + spill ⇒ 倾斜或单分区太大；慢 + GC ⇒ 内存；快但很多 ⇒ 分区数不够 |

再往下分叉就只看一件事 —— **那个 Exchange 的下游算子是谁**：

- `HashAggregate`（聚合倾斜） → 根因是"一个 key 的行数太多" → **解法只有加盐/两阶段**（本篇③A）。
  验证方法：`hashpartitioning_of()` 看分区键；把 key 的 top-N 数出来（`groupBy(k).count().orderBy(desc)`）；
  改完加盐要**先证明答案没变**（④ 那三个集合差）。
- `SortMergeJoin`（join 倾斜） → 根因是"热点 key 在两侧都进同一个桶" →
  **广播小表**（③B，`BroadcastHashJoin` 不按 key 分区）／**手动拆热点 key 再 union**／
  AQE skew join（要够大才触发，`forceOptimizeSkewedJoin` 本镜像实测 `false`）。
  验证方法：两种阈值各取一次计划对比算子名（③B 那段），并检查小表是不是真的还小（行数 × 平均字节）。

### 5.2 参数表（默认值那一列是本镜像 `SET -v` 实测，不是背的）

"实测"这两个字在这里有具体所指：**这张表里敢写值的每一行，都能在盘面上找到打印它的那一行** ——
出厂值由最前面那个单元的 `CITE` 清单打印，AQE 那几项由解法 C 的 `SKEW_KEYS` 清单打印。
往这张表里加一个键，就要同时把它加进那两个清单之一（复审 M4 立的规矩：
只写"实测"而没有对应打印，等于把作者的记忆当读者的证据）。

| 参数 | 本镜像生效值 | 什么时候动 | 动错的表征 |
| --- | --- | --- | --- |
| `spark.sql.shuffle.partitions` | 出厂 200，本篇改 8 | 只影响"每个 task 多大"；**不解决热点 key**（本篇 C 组：8→100，最大桶 47.5%→40.6%） | 调太小 → 单 task 太大、spill、OOM；调太大 → 几千个几 KB 的 task，启动开销主导，且小文件爆炸 |
| `spark.sql.adaptive.enabled` | 出厂 `true` | 生产默认开着；**做度量/教学时关掉**，否则桶数会被改写（实测 200 个 shuffle 分区 → 2 个文件） | 关掉后失去分区合并与 skew join 改写；开着时"你设的 200"只是上限不是结果 |
| `spark.sql.adaptive.coalescePartitions.minPartitionSize` | `1MB` | 想阻止过度合并（小分区也要留给下游并行度）时调 | 调大 → 并行度被压没；调小 → 上万个碎分区 |
| `spark.sql.adaptive.advisoryPartitionSizeInBytes` | 出厂值就是最前面那个单元 `CITE` 打印的那一行（实测）；**跨版本有没有改过名：未逐版本核实**，迁移以官方配置页为准 | join 侧的目标分区大小；要和下面那行 `parallelismFirst` 一起看才解释得通 | 与 minPartitionSize 混为一谈 → "为什么合并不动了" |
| `spark.sql.adaptive.coalescePartitions.parallelismFirst` | 出厂值就是解法 C 那段 `SET -v` 打印的那一行（实测） | 想让"合并小分区"只看目标大小、不被并行度牵着走时改它 | 只调 `advisoryPartitionSizeInBytes` 却忘了它：这一项决定"并行度优先还是大小优先"，于是"目标大小"看着像没生效 |
| `spark.sql.adaptive.skewJoin.enabled` | `true` | 有 join 倾斜且单分区能过阈值时才有意义 | 以为它管聚合 → "为什么 AQE 开了还是慢"（本篇最主要的那条误解） |
| `...skewJoin.skewedPartitionFactor` / `...ThresholdInBytes` | `5.0` / **`256MB`**（实测，解法 C 那段打印；**网上大量资料写 512MB，那是哪个版本的默认值本篇没逐版本核实**） | 小数据集群上想让它真触发，就降阈值 | 阈值降太低 → 正常波动被判成倾斜，join 被拆成一堆小任务反而更慢 |
| `spark.sql.adaptive.forceOptimizeSkewedJoin` | `false`（实测，解法 C 那段打印） | 你确认"AQE 因为'给孩子追加一次 shuffle'而跳过了某次 skew 改写"、且宁愿多一次 shuffle 也要把它拆开时开 | **方向别记反**：默认 `false` 的含义是"宁可漏改这次倾斜，也不给子 stage 追加 shuffle"，所以**开了不会漏优化，代价是多一次 shuffle**；写反就会让人以为"开了才有优化"（评审 M3 订正的就是这句）。语义按官方配置文档，**本镜像只实测过默认值 `false` 这一个数**。它不是"让 skew join 治聚合"的开关 —— 聚合那条没人改写 |
| `spark.sql.autoBroadcastJoinThreshold` | `10MB`（`CITE` 打印） | 想强制/禁止广播 join（③B 那段两头都调过）；`-1` 关自动广播 | 设太大 → build 侧撑爆内存 / `broadcastTimeout`（`CITE` 打印的那个出厂值）到点；广播前没收敛维度 → 行数膨胀（`bd-pyspark-0002`） |
| `spark.sql.files.maxPartitionBytes` | `128MB`（`CITE` 打印） | 读侧合并/切分，本篇不展开（篇 2 的主题） | 与 `openCostInBytes` 一起决定"多少个文件进一个 task" |

**聚合加盐的通用形状（背这一句就够）**：
`groupBy(k, salt).agg(局部可分解聚合)` → `groupBy(k).agg(把部分结果合并的那种聚合)`。
`sum→sum`、`count→sum`、`min/max→min/max`、`avg→(sum,count) 再相除`、
`精确 count(distinct) → 不能这么拆，改两阶段去重`，而**去重那一层的分组键是 `(维度, distinct_col)`、
盐不许进去**（进去了第二阶段 `count()` 就把同一个值数至多 SALT 次 ⇒ 系统性高估；要真加盐得换 `count_distinct`，
本篇没跑那个形状）。

### 5.3 面试官追问链（第三条是机制层，本篇的落点）

1. **"你怎么确认它是倾斜而不是单纯数据量大？"**
   看桶的行数/字节分布的最大值 vs 1/分区数；单纯量大是所有桶一起大。
   追问：**"那你怎么排除长尾 GC？"** —— 看 UI 的 GC time 与 spill，倾斜的 task 是 *input records* 就多，GC 型是 *records* 正常而时间长。
2. **"count(distinct) 为什么更容易倾斜？怎么改？"**
   因为 distinct 那一列本身就是 hash 分区键，一个值全落一个桶；且"去重"不可分解，
   加盐两阶段直接套上去会算错。改法是先 `groupBy(维度, distinct_col)` 把去重做完（这一步天然可并行），
   再 `groupBy(维度).count()`；口径要"近似"时换 `approxCountDistinct`（HLL，误差 `rsd` 可调）。
   追问：**"空值呢？"** —— `null`/空串常常就是那个热点（未登录流量、`shop_id=0`）：
   先 `filter(col.isNotNull())` 单独统计再 union 回来，或给 null 行加随机盐（**只对计数类安全**，
   拿 null 去 join 时加盐会把匹配关系打散，那是另一种错法）。
3. **"为什么 skew hint / AQE skew join 救不了聚合？"**（机制层）
   hash 分区把一个 key 的行**唯一地**送进一个桶，这是分组语义的一部分；把一个桶拆成两份，
   每个 task 就无法判断自己的部分结果是不是完整的。join 没有这个约束 —— 倾斜 key 的另一侧
   可以复制多份、把一侧切成几块分别做小 join，结果集不变。所以 Spark 只对 join 做自动改写
   （`OptimizeSkewedJoin`），聚合那条路只能改分区键本身 —— 也就是本篇的加盐。
   追问：**"加盐之后为什么答案一定不变？"** —— 因为 `sum/count` 在整数上是可分解的结合运算，
   两阶段的第二阶段做的就是把同 key 的部分结果按结合律合起来；
   这条**必须实测**（④ 那三个集合差），不能靠"显然"。
4. **"你说加盐有效，那它慢不慢？"**
   诚实的答案：多一次 shuffle 是固定成本，省下的是最大桶的那部分并行度损失。
   本机（`local[2]`）量不到收益、只量得到成本（④ 那一栏的打印就是这个意思）；
   真实集群上要拿 task 时长分布 + spill 才有结论 —— **这一段标未验证**，别把它当结论讲。

### 5.4 本篇没验的东西（照纪律 2 逐条列出）

- 节点级失效模式：`FetchFailedException`、executor OOM、重试风暴 —— `local[2]` 无法产生。
- `hint("skew")` 在**别的版本**上的行为，以及上面这些 AQE 键名/默认值的跨版本史 —— 本镜像只有一个版本
  （最前面那个单元打印的 `spark.version`），没逐版本核实过；要迁移就查官方配置页，别拿本篇当"某某版本改过名"的依据。
- 广播撑爆内存的具体规模、`broadcastTimeout` 的实际表现 —— 512MB driver 下本篇的小维表远不到。
- AQE skew join 在本机**触发**的样子 —— 阈值 256MB，最大桶实测 2.59MB（parquet 落盘字节），差两个数量级。


## 题库指针（同一套口径，不是"再找篇资料"）

- **`bd-pyspark-0001`** ——「匿名流量占九成的 UV：热点 key 拆分 + 两阶段精确去重」。
  这就是本篇面试题的本体：`user_id` 为空/哨兵值占九成 ⇒ 一个桶吃掉绝大多数行。
  答案的两半正好对应本篇 ③A（加盐把行数打散）与 ⑤.3 第 2 条
  （**精确去重不能直接加盐**，要"先 `groupBy(维度, uid)` 去重、再 `groupBy(维度)` 计数"这个两阶段形状）。
  追问往哪走：口径（是否允许近似）→ 为什么 `count(distinct)` 不可分解 → 空 key 要不要单独算。
- **`bd-pyspark-0002`** ——「join 右表多版本导致行数膨胀：先收敛维度再广播」。
  对应本篇 ③B 的那半边：广播是**绕开 join 倾斜**的解，但前提是小表真的还小；
  维度表带多版本时行数先膨胀，广播就变成 OOM。先 `groupBy` 收敛到"一行一键"（take last-touch），
  再广播 —— 顺序反了就是事故。

两份 notebook 之外的东西（小文件与读侧合并 = 篇 2；计划解读与 AQE 三段 = 篇 3）本系列另有专篇。


In [ ]:
# 最后一个单元必须是它：闸门（tutorials.test.ts）跑完这一篇就结束，
# 不往容器里丢活着的 Spark 内核 —— 详见 WI-93 那条"没有守护"的教训的反面。
spark.stop()
print("stopped")
